# Часть B. Окружения Python

Перед запуском выберите GPU: Runtime → Change runtime type → L4 (для этой части её хватает).

**1. Старт сессии** — то же, что шаги 1–4 части A, плюс папка `envs` на Drive для lock-файлов.

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/unlearning_data"
FAST = "/content/fast"

os.makedirs(DRIVE + "/saves", exist_ok=True)
os.makedirs(DRIVE + "/results_raw", exist_ok=True)
os.makedirs(DRIVE + "/envs", exist_ok=True)      # lock-файлы окружений
os.makedirs(FAST + "/hf_home", exist_ok=True)
os.makedirs(FAST + "/models", exist_ok=True)

os.environ["BIG"] = DRIVE
os.environ["HF_HOME"] = FAST + "/hf_home"
os.environ["MODELS"] = FAST + "/models"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("Старт сессии выполнен")

**2. Ставим uv** — установщик пакетов вместо conda: он создаёт отдельные окружения со своим Python. Сначала ячейка убирает настройки Colab, которые направляют uv в системный Python 3.13.

In [ ]:
# Colab задаёт переменные UV_... (ставить пакеты в системный Python 3.13 с его ограничениями версий)
# и PYTHONPATH (подмешивать свои модули). Убираем их, чтобы окружения unl и atk были отдельными.
for name in list(os.environ):
    if name.startswith("UV_") or name == "PYTHONPATH":
        print("Убираю", name, "=", os.environ.pop(name))

!pip install -q uv
!uv --version

**3. Скачиваем OpenUnlearning** ровно той версии, с которой сверен план (коммит `4ad738a`), и связываем его папку `saves` с Drive. Должно напечататься `4ad738a` и дата 18 марта 2026.

In [ ]:
%%bash
set -e                                   # остановиться на первой ошибке
rm -rf /content/sandbox/open-unlearning  # старая копия, если есть (данные на Drive не трогаются)
mkdir -p /content/sandbox
cd /content/sandbox
git clone -q https://github.com/locuslab/open-unlearning.git
cd open-unlearning
git checkout -q 4ad738aaf60f6a4385f6e2506d01da99e76c31f3
git log -1 --format='%h %cd'
ln -s $BIG/saves saves                   # результаты OpenUnlearning сразу попадают на Drive
ls -l saves

**4. Создаём окружение `unl` и ставим зависимости OpenUnlearning** (Python 3.11, torch 2.4.1, transformers 4.51.3). Займёт несколько минут. В выводе должно быть `Using Python 3.11… environment at: /content/envs/unl`.

In [ ]:
%%bash
set -e
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
uv pip install ".[lm-eval]"

**5. Ставим FlashAttention-2** — готовую сборку под Python 3.11, torch 2.4 и CUDA 12, поэтому без часовой компиляции.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"

**6. Скачиваем логи оценки готовых моделей** — они нужны для Forget Quality. Файлы попадают в `saves/eval`, то есть на Drive; в конце печатается список папок `tofu_…`.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
cd /content/sandbox/open-unlearning
python setup_data.py --eval_logs
ls saves/eval

**7. Проверяем `unl`.** Должно напечататься `2.4.1 12.1 True <ваша GPU>` и `4.51.3 2.6.3`.

In [ ]:
%%bash
source /content/envs/unl/bin/activate
python -c "import torch, transformers, flash_attn; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0)); print(transformers.__version__, flash_attn.__version__)"

**8. Записываем lock-файл `unl`** — точные версии всех пакетов, файл `envs/requirements-unl.lock` на Drive. FlashAttention и сам OpenUnlearning в него не входят: они ставятся отдельно.

In [ ]:
%%bash
set -e
source /content/envs/unl/bin/activate
uv pip freeze | grep -v -e flash-attn -e open-unlearning > $BIG/envs/requirements-unl.lock
wc -l $BIG/envs/requirements-unl.lock    # сколько пакетов записано

**9. Создаём окружение `atk` и ставим vLLM, оценщик и пакеты для анализа.** Займёт несколько минут. В конце должно напечататься `All installed packages are compatible`.

In [ ]:
%%bash
set -e
uv venv /content/envs/atk --python 3.11 --seed --clear
source /content/envs/atk/bin/activate
# vLLM 0.19.1 — последняя версия, собранная под CUDA 12: работает и на драйверах с CUDA 12, и с CUDA 13.
# cupy-cuda12x (spaCy на GPU) ставится отдельно, а не через spacy[cuda12x]: иначе uv откатит vLLM к старой версии.
# Последняя строка — модель spaCy en_core_web_trf 3.8.0 (ту же ставит команда spacy download).
uv pip install "vllm==0.19.1" openai spacy cupy-cuda12x bert-score sentence-transformers rapidfuzz \
    pandas pyarrow orjson zstandard hydra-core tenacity \
    scipy statsmodels scikit-learn lifelines matplotlib \
    pytest hypothesis ruff mypy \
    "en_core_web_trf @ https://github.com/explosion/spacy-models/releases/download/en_core_web_trf-3.8.0/en_core_web_trf-3.8.0-py3-none-any.whl"
uv pip check

**10. Записываем скрипт проверки `atk`.**

In [ ]:
%%writefile /content/check_atk.py
import torch
import vllm

print("torch", torch.__version__, "| CUDA доступна:", torch.cuda.is_available(), "| vLLM", vllm.__version__)

import spacy

print("spaCy на GPU:", spacy.prefer_gpu())
nlp = spacy.load("en_core_web_trf")
doc = nlp("Basil Mahfouz Al-Kuwaiti was born in Kuwait City in 1956.")
for entity in doc.ents:
    print("   ", entity.text, "—", entity.label_)

from bert_score import BERTScorer

scorer = BERTScorer(lang="en", model_type="roberta-large", rescale_with_baseline=True)
precision, recall, f1 = scorer.score(["He was born in Kuwait."], ["The author was born in Kuwait City."])
print("BERTScore F1:", round(f1.item(), 3))

**11. Проверяем `atk`.** Должно быть: версия vLLM, `CUDA доступна: True`, `spaCy на GPU: True`, сущности с метками PERSON, GPE и DATE и одно число BERTScore F1. При первом запуске BERTScore скачивает модель roberta-large (около 1,4 ГБ).

In [ ]:
%%bash
source /content/envs/atk/bin/activate
vllm --version
python /content/check_atk.py

**12. Записываем lock-файл `atk`** — файл `envs/requirements-atk.lock` на Drive.

In [ ]:
%%bash
set -e
source /content/envs/atk/bin/activate
uv pip freeze > $BIG/envs/requirements-atk.lock
wc -l $BIG/envs/requirements-atk.lock    # сколько пакетов записано

**13. Записываем окружения в журнал** (`journal.md` на Drive).

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# главные версии — из lock-файлов
unl = !grep -E "^(torch|transformers|accelerate|deepspeed)==" $BIG/envs/requirements-unl.lock
atk = !grep -E "^(torch|vllm|transformers|spacy|cupy-cuda12x)==" $BIG/envs/requirements-atk.lock
unl_versions = ", ".join(unl)
atk_versions = ", ".join(atk)
today = datetime.now(ZoneInfo("Europe/Moscow")).date()

text = f"""
## {today} — Окружения unl и atk (Colab)
- OpenUnlearning: коммит 4ad738a
- unl: Python 3.11, {unl_versions}, flash-attn 2.6.3
- atk: Python 3.11, {atk_versions}
- Lock-файлы: {DRIVE}/envs/requirements-unl.lock и requirements-atk.lock
"""

with open(DRIVE + "/journal.md", "a", encoding="utf-8") as f:
    f.write(text)

print(text)

**14. Проверяем, что окружения собираются из lock-файлов** (блок 9 плана). Runtime → Disconnect and delete runtime, затем выполните шаги 1 и 2, эту ячейку и проверки 7, 10 и 11. Так же окружения будут собираться в начале следующих частей.

In [ ]:
%%bash
set -e
uv venv /content/envs/unl --python 3.11 --seed --clear
source /content/envs/unl/bin/activate
uv pip install -r $BIG/envs/requirements-unl.lock
uv pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.6.3/flash_attn-2.6.3+cu123torch2.4cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
deactivate

uv venv /content/envs/atk --python 3.11 --seed --clear
source /content/envs/atk/bin/activate
uv pip install -r $BIG/envs/requirements-atk.lock
uv pip check